In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import Normalize
import pickle

# --- 数据加载和处理 ---
with open('../../data/supplementary_figures/slr_res/DIVA_data/Coast_with_Mang.pkl', 'rb') as f:
    data = pickle.load(f)

# 提取所需数据
agb_present = data['agb_present']
agb_126 = data['agb_126']
agb_585 = data['agb_585']
area = data['area']
lat_lon_coast = data['lat_lon_coast']
area_restoration_126 = data['area_restoration_126'][:, 0]
area_restoration_585 = data['area_restoration_585'][:, 0]

lat = lat_lon_coast[:, 0]
lon = lat_lon_coast[:, 1]

# 计算恢复贡献的百分比
with np.errstate(divide='ignore', invalid='ignore'):
    Res_change_Percent_126 = np.where(
        (agb_present * area) != 0,
        (agb_126 * area_restoration_126) / (agb_present * area),
        np.nan
    )
    Res_change_Percent_585 = np.where(
        (agb_present * area) != 0,
        (agb_585 * area_restoration_585) / (agb_present * area),
        np.nan
    )

# --- 关键改动：数据对齐和差值计算 ---
# 先将两个情景的数据合并，再统一清理NaN，确保点位一一对应
data_combined = np.column_stack((lat, lon, Res_change_Percent_126, Res_change_Percent_585))

# 清理在任一情景中为NaN的行
data_clean = data_combined[~np.isnan(data_combined[:, 2:]).any(axis=1)]

# 提取对齐后的干净数据
lat_c = data_clean[:, 0]
lon_c = data_clean[:, 1]
Res_change_Percent_126_c = data_clean[:, 2] * 100
Res_change_Percent_585_c = data_clean[:, 3] * 100

# 计算差值 (a - b)
Res_change_Percent_diff = Res_change_Percent_126_c - Res_change_Percent_585_c


# --- 绘图部分 ---

# 设置全局字体
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'

# 创建Figure和Gridspec，布局为3行1列
fig = plt.figure(figsize=(12, 9), constrained_layout=True)
grid = fig.add_gridspec(3, 1)

# 定义色带和标准化范围
# a, b图的色带
greens_cmap = plt.get_cmap('Greens')
greens_norm = Normalize(vmin=0, vmax=40) # 范围0-50%

# c图的色带 (差值)
diff_cmap = plt.get_cmap('OrRd')
# 使颜色范围关于0对称
max_abs_diff = np.max(np.abs(Res_change_Percent_diff))
diff_norm = Normalize(vmin=0, vmax=20)


# --- 绘制第一张图 (a): SSP126 ---
ax1 = fig.add_subplot(grid[0], projection=ccrs.PlateCarree())
ax1.set_extent([-180, 180, -40, 40])
ax1.set_title('a', loc='left', fontweight='bold')
ax1.text(-175, 35, 'Low-warming', ha='left', va='top')
ax1.coastlines(resolution='110m', linewidth=0.75, color='black', zorder=10)
scatter1 = ax1.scatter(lon_c, lat_c, c=Res_change_Percent_126_c, cmap=greens_cmap, norm=greens_norm, s=10)

# --- 绘制第二张图 (b): SSP585 ---
ax2 = fig.add_subplot(grid[1], projection=ccrs.PlateCarree())
ax2.set_extent([-180, 180, -40, 40])
ax2.set_title('b', loc='left', fontweight='bold')
ax2.text(-175, 35, 'High-warming', ha='left', va='top')
ax2.coastlines(resolution='110m', linewidth=0.75, color='black', zorder=10)
scatter2 = ax2.scatter(lon_c, lat_c, c=Res_change_Percent_585_c, cmap=greens_cmap, norm=greens_norm, s=10)

# --- 绘制第三张图 (c): Difference (a - b) ---
ax3 = fig.add_subplot(grid[2], projection=ccrs.PlateCarree())
ax3.set_extent([-180, 180, -40, 40])
ax3.set_title('c', loc='left', fontweight='bold')
ax3.text(-175, 35, 'Low-warming minus\nHigh-warming', ha='left', va='top')
ax3.coastlines(resolution='110m', linewidth=0.75, color='black', zorder=10)
scatter3 = ax3.scatter(lon_c, lat_c, c=Res_change_Percent_diff, cmap=diff_cmap, norm=diff_norm, s=10, alpha=0.7)

# --- 添加颜色条 ---

# 为 a 图添加颜色条
cbar_ax_1 = fig.add_axes([0.06, 0.72, 0.15, 0.02])
cbar_1 = plt.colorbar(scatter1, cax=cbar_ax_1, orientation='horizontal')
cbar_1.ax.set_title('Relative AGB change\nfrom Restoration (%)', fontsize=14, pad=10)
cbar_1.set_ticks([0, 20, 40])
cbar_1.ax.tick_params(labelsize=14, length=0)
cbar_1.outline.set_visible(False)

# 为 b 图添加颜色条
cbar_ax_2 = fig.add_axes([0.06, 0.38, 0.15, 0.02])
cbar_2 = plt.colorbar(scatter2, cax=cbar_ax_2, orientation='horizontal')
cbar_2.ax.set_title('Relative AGB change\nfrom Restoration (%)', fontsize=14, pad=10)
cbar_2.set_ticks([0, 20, 40])
cbar_2.ax.tick_params(labelsize=14, length=0)
cbar_2.outline.set_visible(False)

# 为 c 图添加颜色条
cbar_ax_3 = fig.add_axes([0.06, 0.05, 0.15, 0.02])
cbar_3 = plt.colorbar(scatter3, cax=cbar_ax_3, orientation='horizontal')
cbar_3.ax.set_title('Difference (%)', fontsize=14, pad=10)
cbar_3.set_ticks([0, 10, 20])
cbar_3.ax.tick_params(labelsize=14, length=0)
cbar_3.outline.set_visible(False)

# 保存和显示图像
plt.savefig('../../figures/supplementary/figS10_restoration_relative_change_distribution.png', dpi=800, bbox_inches='tight')
plt.show()